In [ ]:
# Imports
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)


# Linear Regression, Regularization and Testing Exercise
## Big Mart Sales Prediction

In this notebook I work with the **Big Mart Sales Prediction** dataset.  
The target variable is `Item_Outlet_Sales`, so this is a supervised regression problem.

The solution follows the exercise structure: research question, EDA, preprocessing, train/test split, linear regression baseline, evaluation, another regression model, hyperparameter tuning, comparison, and one improved preprocessing iteration.


## Problem 1. Research question

**Business question:** Can I predict the sales of a product in a specific outlet using available product and store information? A useful model can help the business estimate expected sales, plan inventory, and understand which factors are associated with higher or lower sales.

**Data science question:** Can a regression model predict `Item_Outlet_Sales` with acceptable error on unseen data?

**Target:** `Item_Outlet_Sales`  
**Main evaluation metrics:** MAE, RMSE, and R².  
**Expected result:** A simple linear regression should provide a baseline, while a more flexible model may capture non-linear relationships and improve RMSE/R².


In [ ]:
research_question = "Predict Item_Outlet_Sales using product and outlet attributes."
metrics = ["MAE", "RMSE", "R2"]
print(research_question)
print("Evaluation metrics:", metrics)


## Problem 2. Read and explore the dataset

I first load `train.csv`, inspect the shape, missing values, data types, summary statistics, and a few distributions. This is important before choosing preprocessing and models.


In [ ]:
def find_data_file(filename: str) -> Path:
    """Find a data file in common notebook locations."""
    candidates = [
        Path(filename),
        Path("data") / filename,
        Path(".") / filename,
        Path("/mnt/data") / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {filename}. Put it in the same folder as this notebook or in a data/ folder."
    )

train_path = find_data_file("train.csv")
df = pd.read_csv(train_path)

print("Loaded:", train_path)
print("Shape:", df.shape)
display(df.head())


In [ ]:
print("Data types:")
display(df.dtypes.to_frame("dtype"))

print("Missing values:")
display(df.isna().sum().to_frame("missing_count"))

print("Numeric summary:")
display(df.describe())

print("Categorical summary:")
display(df.describe(include="object"))


In [ ]:
# Basic target distribution
plt.figure(figsize=(8, 4))
plt.hist(df["Item_Outlet_Sales"], bins=40)
plt.title("Distribution of Item_Outlet_Sales")
plt.xlabel("Item_Outlet_Sales")
plt.ylabel("Count")
plt.show()

# Item MRP is expected to be an important numerical predictor.
plt.figure(figsize=(8, 4))
plt.scatter(df["Item_MRP"], df["Item_Outlet_Sales"], alpha=0.3)
plt.title("Item_MRP vs Item_Outlet_Sales")
plt.xlabel("Item_MRP")
plt.ylabel("Item_Outlet_Sales")
plt.show()


In [ ]:
# Quick categorical checks
for col in ["Item_Fat_Content", "Outlet_Size", "Outlet_Type", "Outlet_Location_Type"]:
    print("\n", col)
    display(df[col].value_counts(dropna=False).to_frame("count"))


### EDA notes

Important observations:

- `Item_Weight` and `Outlet_Size` contain missing values.
- `Item_Fat_Content` has inconsistent labels such as `LF`, `low fat`, and `reg`.
- `Item_Visibility` has many zero values, which may be suspicious because a product usually has some visibility.
- `Item_MRP` appears strongly related to sales, so it should be useful for prediction.
- There are both numerical and categorical variables, so preprocessing is required before using scikit-learn models.


## Problem 3. Preprocessing

Scikit-learn models need numerical input. I use two preprocessing pipelines:

1. **Linear model pipeline:** median imputation + scaling for numeric columns, most-frequent imputation + one-hot encoding for categorical columns.
2. **Tree model pipeline:** median imputation for numeric columns and ordinal encoding for categorical columns.

The linear model gets one-hot encoding because it should not treat categories as ordered numbers. The tree model uses ordinal encoding for a compact representation.


In [ ]:
target = "Item_Outlet_Sales"
X = df.drop(columns=[target])
y = df[target]

numeric_features = [
    "Item_Weight",
    "Item_Visibility",
    "Item_MRP",
    "Outlet_Establishment_Year",
]

categorical_features = [
    "Item_Identifier",
    "Item_Fat_Content",
    "Item_Type",
    "Outlet_Identifier",
    "Outlet_Size",
    "Outlet_Location_Type",
    "Outlet_Type",
]

numeric_linear = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_linear = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess_linear = ColumnTransformer(transformers=[
    ("num", numeric_linear, numeric_features),
    ("cat", categorical_linear, categorical_features),
])

numeric_tree = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_tree = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocess_tree = ColumnTransformer(transformers=[
    ("num", numeric_tree, numeric_features),
    ("cat", categorical_tree, categorical_features),
])

print("Preprocessing pipelines created.")


## Problem 4. Train/test split and validation

I split the data before fitting preprocessing pipelines. This prevents data leakage because imputers, scalers, and encoders are fitted only on the training data.

I use an 80/20 split and a fixed random state for reproducibility. Cross-validation is also used later to check whether results are stable.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## Problem 5. Baseline model: Linear Regression

The baseline model is standard `LinearRegression`. This gives a clear reference point before trying more flexible models.


In [ ]:
linear_model = Pipeline(steps=[
    ("preprocess", preprocess_linear),
    ("model", LinearRegression()),
])

linear_model.fit(X_train, y_train)
print("Linear Regression baseline trained.")


## Problem 6. Evaluation

I evaluate models using:

- **MAE:** average absolute error; easy to interpret in sales units.
- **RMSE:** penalizes large errors more strongly.
- **R²:** proportion of variance explained by the model.

I also include visual evaluation with actual vs. predicted values and residuals.


In [ ]:
def evaluate_regression_model(name, model, X_test, y_test):
    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)
    return {
        "model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    }, predictions

linear_metrics, linear_predictions = evaluate_regression_model(
    "Linear Regression", linear_model, X_test, y_test
)

evaluation_results = pd.DataFrame([linear_metrics])
display(evaluation_results)


In [ ]:
# Visual evaluation: actual vs predicted
plt.figure(figsize=(6, 6))
plt.scatter(y_test, linear_predictions, alpha=0.35)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()])
plt.title("Linear Regression: Actual vs Predicted")
plt.xlabel("Actual sales")
plt.ylabel("Predicted sales")
plt.show()

# Residual plot
linear_residuals = y_test - linear_predictions
plt.figure(figsize=(8, 4))
plt.scatter(linear_predictions, linear_residuals, alpha=0.35)
plt.axhline(0)
plt.title("Linear Regression: Residuals")
plt.xlabel("Predicted sales")
plt.ylabel("Residual")
plt.show()


In [ ]:
# Cross-validation for the baseline
cv_scores = cross_validate(
    linear_model,
    X,
    y,
    cv=5,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2",
    },
)

cv_summary = pd.DataFrame({
    "metric": ["MAE", "RMSE", "R2"],
    "mean": [
        -cv_scores["test_mae"].mean(),
        -cv_scores["test_rmse"].mean(),
        cv_scores["test_r2"].mean(),
    ],
    "std": [
        cv_scores["test_mae"].std(),
        cv_scores["test_rmse"].std(),
        cv_scores["test_r2"].std(),
    ],
})

display(cv_summary)


### Baseline discussion

The linear model is useful as a baseline because it is simple and interpretable. However, sales can depend on non-linear interactions between item price, outlet type, outlet age, product type, and visibility. For that reason, I expect a non-linear model to perform better.


## Problem 7. A different regression model

For the second model I use a `DecisionTreeRegressor`. It can capture non-linear relationships and feature interactions without requiring the exact linear form of the relationship.


In [ ]:
tree_model = Pipeline(steps=[
    ("preprocess", preprocess_tree),
    ("model", DecisionTreeRegressor(
        random_state=RANDOM_STATE,
        max_depth=6,
        min_samples_leaf=20,
    )),
])

tree_model.fit(X_train, y_train)
tree_metrics, tree_predictions = evaluate_regression_model(
    "Decision Tree", tree_model, X_test, y_test
)

evaluation_results = pd.DataFrame([linear_metrics, tree_metrics])
display(evaluation_results)


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, tree_predictions, alpha=0.35)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()])
plt.title("Decision Tree: Actual vs Predicted")
plt.xlabel("Actual sales")
plt.ylabel("Predicted sales")
plt.show()


## Problem 8. Hyperparameter tuning

For the decision tree, I tune two important hyperparameters:

- `max_depth`: controls how complex the tree can become.
- `min_samples_leaf`: controls the minimum number of samples required in a leaf and helps reduce overfitting.

I use grid search with cross-validation and RMSE as the main optimization metric.


In [ ]:
param_grid = {
    "model__max_depth": [4, 5, 6, 7],
    "model__min_samples_leaf": [10, 20, 40],
}

tree_for_tuning = Pipeline(steps=[
    ("preprocess", preprocess_tree),
    ("model", DecisionTreeRegressor(random_state=RANDOM_STATE)),
])

grid_search = GridSearchCV(
    estimator=tree_for_tuning,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV RMSE:", -grid_search.best_score_)

best_tree_model = grid_search.best_estimator_
best_tree_metrics, best_tree_predictions = evaluate_regression_model(
    "Tuned Decision Tree", best_tree_model, X_test, y_test
)

evaluation_results = pd.DataFrame([linear_metrics, tree_metrics, best_tree_metrics])
display(evaluation_results)


## Problem 9. Evaluate and iterate

Here I compare the baseline and the tuned model using the same metrics. A good result is not just a lower test RMSE once; the model should also have reasonable cross-validation performance.


In [ ]:
comparison = pd.DataFrame([linear_metrics, tree_metrics, best_tree_metrics]).sort_values("RMSE")
display(comparison)

best_model_name = comparison.iloc[0]["model"]
print(f"Best model on the test split by RMSE: {best_model_name}")

# Cross-validation for the tuned model
best_tree_cv = cross_validate(
    best_tree_model,
    X,
    y,
    cv=5,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2",
    },
)

best_tree_cv_summary = pd.DataFrame({
    "metric": ["MAE", "RMSE", "R2"],
    "mean": [
        -best_tree_cv["test_mae"].mean(),
        -best_tree_cv["test_rmse"].mean(),
        best_tree_cv["test_r2"].mean(),
    ],
    "std": [
        best_tree_cv["test_mae"].std(),
        best_tree_cv["test_rmse"].std(),
        best_tree_cv["test_r2"].std(),
    ],
})

display(best_tree_cv_summary)


### Model comparison discussion

The linear regression model is easier to explain, but it may miss non-linear patterns. The decision tree can capture interactions such as different outlet types having different sales patterns at different price levels. If the tuned tree improves RMSE and R², it is more useful for prediction, but it is still less stable than a regularized linear model and can overfit if it becomes too deep.

A production-quality solution could also test random forests, gradient boosting, log-transforming the target, and more careful feature engineering.


## Problem 10. Change preprocessing and try again

For the final iteration I change preprocessing in a meaningful way:

- Standardize `Item_Fat_Content` labels.
- Treat zero `Item_Visibility` as missing and impute it.
- Add `Outlet_Age` instead of using only the raw establishment year.
- Add `Item_Code`, the first two characters of `Item_Identifier`, as a compact product category signal.

I then train a regularized linear model (`Ridge`) and compare it with previous results.


In [ ]:
def clean_bigmart_features(X_input):
    X_clean = X_input.copy()

    X_clean["Item_Fat_Content"] = X_clean["Item_Fat_Content"].replace({
        "LF": "Low Fat",
        "low fat": "Low Fat",
        "reg": "Regular",
    })

    X_clean["Item_Visibility"] = X_clean["Item_Visibility"].replace(0, np.nan)
    X_clean["Outlet_Age"] = 2013 - X_clean["Outlet_Establishment_Year"]
    X_clean["Item_Code"] = X_clean["Item_Identifier"].astype(str).str[:2]

    return X_clean

engineered_numeric_features = [
    "Item_Weight",
    "Item_Visibility",
    "Item_MRP",
    "Outlet_Age",
]

engineered_categorical_features = [
    "Item_Identifier",
    "Item_Code",
    "Item_Fat_Content",
    "Item_Type",
    "Outlet_Identifier",
    "Outlet_Size",
    "Outlet_Location_Type",
    "Outlet_Type",
]

engineered_preprocess = Pipeline(steps=[
    ("clean_features", FunctionTransformer(clean_bigmart_features, validate=False)),
    ("preprocess", ColumnTransformer(transformers=[
        ("num", numeric_linear, engineered_numeric_features),
        ("cat", categorical_linear, engineered_categorical_features),
    ])),
])

ridge_model = Pipeline(steps=[
    ("preprocess", engineered_preprocess),
    ("model", Ridge(alpha=10.0, random_state=RANDOM_STATE)),
])

ridge_model.fit(X_train, y_train)
ridge_metrics, ridge_predictions = evaluate_regression_model(
    "Ridge + Engineered Features", ridge_model, X_test, y_test
)

final_results = pd.DataFrame([
    linear_metrics,
    tree_metrics,
    best_tree_metrics,
    ridge_metrics,
]).sort_values("RMSE")

display(final_results)


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, ridge_predictions, alpha=0.35)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()])
plt.title("Ridge + Engineered Features: Actual vs Predicted")
plt.xlabel("Actual sales")
plt.ylabel("Predicted sales")
plt.show()


## Final conclusion

The notebook starts with a simple linear regression baseline and then tests a more flexible tree-based model. The final preprocessing iteration tries to clean inconsistent values and add simple domain-driven features. The best model should be selected by comparing MAE, RMSE, R², visual residual behavior, and cross-validation stability.

From a business perspective, lower RMSE/MAE means more accurate sales estimates. From a data science perspective, the main lesson is that preprocessing decisions and validation strategy matter as much as the choice of model.
